
# MedQuAD RAG: Retrieve + Generate with an 8B Instruct Model



```text
/content/drive/MyDrive/vector store/
├── medquad_medcpt.faiss
├── medquad_metadata.pkl
└── medquad_vectorstore_config.json
```

### Generation rules

The model is instructed to:

- answer **only from the retrieved evidence**;
- not invent unsupported medical facts;
- explicitly say when the retrieved evidence is insufficient;
- cite supporting evidence using `[Chunk 1]`, `[Chunk 2]`, etc.;
- keep each query independent so prior questions do not contaminate evaluation.

##### dependencies

In [1]:
!pip -q install -U transformers accelerate bitsandbytes faiss-cpu sentencepiece safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 61.0 MB/s eta 0:00:00


## Imports and configuration

In [2]:
import os
import json
import pickle
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import faiss

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# ---------------------------------------------------------
# Vector-store configuration
# ---------------------------------------------------------
VECTOR_STORE_DIR = Path("/content/drive/MyDrive/vector store")
INDEX_PATH = VECTOR_STORE_DIR / "medquad_medcpt.faiss"
METADATA_PATH = VECTOR_STORE_DIR / "medquad_metadata.pkl"
CONFIG_PATH = VECTOR_STORE_DIR / "medquad_vectorstore_config.json"

QUERY_MODEL_NAME = "ncbi/MedCPT-Query-Encoder"

# Qwen3-8B is used in non-thinking mode for the instruct-side experiment.
GENERATOR_MODEL_NAME = "Qwen/Qwen3-8B"

TOP_K = 5
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0

# 4-bit loading is recommended for a standard Colab T4/L4 GPU.
LOAD_IN_4BIT = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU memory: {total_gb:.1f} GB")
else:
    print(
        "WARNING: No CUDA GPU detected. An 8B model will be very slow on CPU. "
        "In Colab, choose Runtime → Change runtime type → GPU."
    )

PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4
GPU memory: 14.6 GB


## Mount Google Drive

In [3]:
from google.colab import drive

drive.mount("/content/drive")

print("Vector-store directory:", VECTOR_STORE_DIR)

Mounted at /content/drive
Vector-store directory: /content/drive/MyDrive/vector store


## Verify the saved vector-store files

In [4]:
required_files = [INDEX_PATH, METADATA_PATH]

missing = [str(p) for p in required_files if not p.exists()]

if missing:
    raise FileNotFoundError(
        "The following required vector-store files were not found:\n"
        + "\n".join(missing)
        + "\n\nRun the MedQuAD vector-store creation notebook first."
    )

print("Required files found:")
for p in required_files:
    print(f"✓ {p}")

if CONFIG_PATH.exists():
    print(f"✓ {CONFIG_PATH}")
else:
    print(
        "Note: medquad_vectorstore_config.json was not found. "
        "Retrieval can still run using the configured MedCPT query encoder."
    )

Required files found:
✓ /content/drive/MyDrive/vector store/medquad_medcpt.faiss
✓ /content/drive/MyDrive/vector store/medquad_metadata.pkl
✓ /content/drive/MyDrive/vector store/medquad_vectorstore_config.json


## Load FAISS index and chunk metadata

In [5]:
faiss_index = faiss.read_index(str(INDEX_PATH))

with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)

print("FAISS vectors:", faiss_index.ntotal)
print("Metadata records:", len(metadata))
print("FAISS dimension:", faiss_index.d)

if faiss_index.ntotal != len(metadata):
    raise ValueError(
        "FAISS vector count does not match metadata record count. "
        "The index and metadata files may be from different builds."
    )

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        vectorstore_config = json.load(f)

    print("\nVector-store config:")
    print(json.dumps(vectorstore_config, indent=2))

FAISS vectors: 18142
Metadata records: 18142
FAISS dimension: 768

Vector-store config:
{
  "dataset": "pythonafroz/medquad-medical-question-answer-for-ai-research",
  "answer_column": "answer",
  "question_column": "question",
  "source_column": "source",
  "focus_column": "focus_area",
  "question_type_column": null,
  "article_embedding_model": "ncbi/MedCPT-Article-Encoder",
  "query_embedding_model": "ncbi/MedCPT-Query-Encoder",
  "chunk_size_tokens": 500,
  "chunk_overlap_tokens": 50,
  "distance": "cosine similarity via normalized IndexFlatIP",
  "embedding_dimension": 768,
  "number_of_vectors": 18142
}



## Load the MedCPT Query Encoder

The query embedding is L2-normalized before FAISS search, matching the cosine-similarity index created previously.


In [6]:
query_tokenizer = AutoTokenizer.from_pretrained(QUERY_MODEL_NAME)
query_model = AutoModel.from_pretrained(QUERY_MODEL_NAME)

query_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
query_model = query_model.to(query_device)
query_model.eval()

print("Loaded:", QUERY_MODEL_NAME)
print("Query encoder device:", query_device)
print("Hidden size:", query_model.config.hidden_size)

if query_model.config.hidden_size != faiss_index.d:
    raise ValueError(
        f"Query encoder dimension ({query_model.config.hidden_size}) "
        f"does not match FAISS index dimension ({faiss_index.d})."
    )

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded: ncbi/MedCPT-Query-Encoder
Query encoder device: cuda
Hidden size: 768


## Query embedding and top-k retrieval

In [7]:
@torch.inference_mode()
def embed_query(question: str) -> np.ndarray:
    question = str(question).strip()

    if not question:
        raise ValueError("Question cannot be empty.")

    encoded = query_tokenizer(
        [question],
        truncation=True,
        padding=True,
        max_length=64,
        return_tensors="pt",
    )
    encoded = {k: v.to(query_device) for k, v in encoded.items()}

    outputs = query_model(**encoded)

    # MedCPT retrieval uses the [CLS] representation.
    embedding = outputs.last_hidden_state[:, 0, :]
    embedding = embedding.float().cpu().numpy().astype("float32")

    # The saved FAISS IndexFlatIP uses normalized vectors,
    # so this converts inner product into cosine similarity.
    faiss.normalize_L2(embedding)

    return embedding


def retrieve(question: str, top_k: int = TOP_K):
    query_vector = embed_query(question)

    scores, indices = faiss_index.search(query_vector, top_k)

    results = []

    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]),
        start=1,
    ):
        if idx < 0:
            continue

        item = dict(metadata[idx])

        item["rank"] = rank
        item["score"] = float(score)
        item["faiss_id"] = int(idx)

        results.append(item)

    return results

## Helper to display retrieved evidence

In [8]:
def show_retrieved_chunks(results):
    if not results:
        print("No chunks retrieved.")
        return

    print("\n" + "=" * 110)
    print("RETRIEVED EVIDENCE")
    print("=" * 110)

    for r in results:
        print(
            f"\n[Chunk {r['rank']}] "
            f"cosine={r['score']:.4f} | "
            f"row={r.get('row_id')} | "
            f"chunk={r.get('chunk_id')}"
        )

        if r.get("question"):
            print("Original MedQuAD question:", r["question"])

        if r.get("focus_area"):
            print("Focus area:", r["focus_area"])

        if r.get("question_type"):
            print("Question type:", r["question_type"])

        if r.get("source"):
            print("Source:", r["source"])

        print("Text:")
        print(r["text"])

        print("-" * 110)

## Test retrieval before loading the generator

In [9]:
test_question = "What are the symptoms of diabetes?"

test_results = retrieve(test_question, top_k=TOP_K)

print("Question:", test_question)
show_retrieved_chunks(test_results)

Question: What are the symptoms of diabetes?

RETRIEVED EVIDENCE

[Chunk 1] cosine=0.7458 | row=8940 | chunk=0
Original MedQuAD question: What is the outlook for Diabetic Neuropathy ?
Focus area: Diabetic Neuropathy
Source: NINDS
Text:
the prognosis for diabetic neuropathy depends largely on how well the underlying condition of diabetes is handled. treating diabetes may halt progression and improve symptoms of the neuropathy, but recovery is slow. the painful sensations of diabetic neuropathy may become severe enough to cause depression in some patients.
--------------------------------------------------------------------------------------------------------------

[Chunk 2] cosine=0.7412 | row=12867 | chunk=0
Original MedQuAD question: How many people are affected by nephrogenic diabetes insipidus ?
Focus area: nephrogenic diabetes insipidus
Source: GHR
Text:
the prevalence of nephrogenic diabetes insipidus is unknown, although the condition is thought to be rare. the acquired form occ


## Qwen3-8B as the instruct-side generator

The notebook uses:

```text
Qwen/Qwen3-8B
```



In [10]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "A CUDA GPU is strongly recommended for the 8B generator. "
        "In Colab choose Runtime → Change runtime type → GPU, then rerun."
    )

if LOAD_IN_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    generator_model = AutoModelForCausalLM.from_pretrained(
        GENERATOR_MODEL_NAME,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
else:
    generator_model = AutoModelForCausalLM.from_pretrained(
        GENERATOR_MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )

generator_tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL_NAME,
    use_fast=True,
)

if generator_tokenizer.pad_token_id is None:
    generator_tokenizer.pad_token = generator_tokenizer.eos_token

generator_model.eval()

print("Loaded generator:", GENERATOR_MODEL_NAME)
print("4-bit:", LOAD_IN_4BIT)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Loaded generator: Qwen/Qwen3-8B
4-bit: True


##  Evidence-only RAG prompt

In [11]:
SYSTEM_PROMPT = '''You are an evidence-grounded medical information assistant used in a research RAG system.

You must follow these rules:

1. Answer the user's question ONLY using information explicitly supported by the retrieved evidence.
2. Do not add medical facts from your internal knowledge if they are not supported by the retrieved evidence.
3. Every substantive medical statement should be supported by one or more citations in the form [Chunk 1], [Chunk 2], etc.
4. Never cite a chunk that does not support the statement.
5. If the retrieved evidence is insufficient to answer the question reliably, say exactly:
   "The retrieved evidence is insufficient to answer this question reliably."
   You may then briefly state what information is missing, but do not invent an answer.
6. If retrieved chunks conflict, explicitly describe the conflict rather than silently choosing one.
7. Be concise, clinically clear, and directly relevant to the user's question.
8. Do not claim to diagnose the user or replace professional medical care.
'''


def build_context(results):
    context_blocks = []

    for r in results:
        lines = [
            f"[Chunk {r['rank']}]",
            f"Similarity score: {r['score']:.4f}",
        ]

        if r.get("question"):
            lines.append(f"Original MedQuAD question: {r['question']}")

        if r.get("focus_area"):
            lines.append(f"Focus area: {r['focus_area']}")

        if r.get("question_type"):
            lines.append(f"Question type: {r['question_type']}")

        if r.get("source"):
            lines.append(f"Source: {r['source']}")

        lines.append("Evidence:")
        lines.append(str(r["text"]))

        context_blocks.append("\n".join(lines))

    return "\n\n".join(context_blocks)


def build_messages(question, results):
    context = build_context(results)

    user_prompt = f'''USER QUESTION:
{question}

RETRIEVED EVIDENCE:
{context}

TASK:
Answer the USER QUESTION using only the RETRIEVED EVIDENCE.
Use [Chunk n] citations immediately after the claims they support.
If the evidence does not adequately support an answer, use the required insufficient-evidence statement.
'''

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

## Generate the final grounded answer

In [12]:
@torch.inference_mode()
def generate_answer(question, results):
    messages = build_messages(question, results)

    # Qwen3 supports disabling its thinking mode through the chat template.
    try:
        model_inputs = generator_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
            return_tensors="pt",
            return_dict=True,
        )
    except TypeError:
        # Compatibility fallback if a future tokenizer implementation
        # does not expose enable_thinking.
        model_inputs = generator_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )

    generator_device = next(generator_model.parameters()).device
    model_inputs = {
        key: value.to(generator_device)
        for key, value in model_inputs.items()
    }

    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
        "pad_token_id": generator_tokenizer.pad_token_id,
        "eos_token_id": generator_tokenizer.eos_token_id,
    }

    outputs = generator_model.generate(
        **model_inputs,
        **generation_kwargs,
    )

    prompt_len = model_inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][prompt_len:]

    answer = generator_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    ).strip()

    return answer

##  Complete RAG function

In [13]:
def ask_medical_rag(
    question: str,
    top_k: int = TOP_K,
    show_context: bool = True,
):
    question = str(question).strip()

    if not question:
        raise ValueError("Question cannot be empty.")

    print("\n" + "=" * 110)
    print("USER QUESTION")
    print("=" * 110)
    print(question)

    # 1. Retrieve evidence
    results = retrieve(question, top_k=top_k)

    # 2. Display retrieved chunks and similarity scores
    if show_context:
        show_retrieved_chunks(results)

    # 3. Generate evidence-only answer
    answer = generate_answer(question, results)

    print("\n" + "=" * 110)
    print("FINAL RAG ANSWER")
    print("=" * 110)
    print(answer)

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": results,
    }

## Run one complete example

In [14]:
result = ask_medical_rag(
    "What are the symptoms of diabetes?",
    top_k=5,
    show_context=True,
)


USER QUESTION
What are the symptoms of diabetes?

RETRIEVED EVIDENCE

[Chunk 1] cosine=0.7458 | row=8940 | chunk=0
Original MedQuAD question: What is the outlook for Diabetic Neuropathy ?
Focus area: Diabetic Neuropathy
Source: NINDS
Text:
the prognosis for diabetic neuropathy depends largely on how well the underlying condition of diabetes is handled. treating diabetes may halt progression and improve symptoms of the neuropathy, but recovery is slow. the painful sensations of diabetic neuropathy may become severe enough to cause depression in some patients.
--------------------------------------------------------------------------------------------------------------

[Chunk 2] cosine=0.7412 | row=12867 | chunk=0
Original MedQuAD question: How many people are affected by nephrogenic diabetes insipidus ?
Focus area: nephrogenic diabetes insipidus
Source: GHR
Text:
the prevalence of nephrogenic diabetes insipidus is unknown, although the condition is thought to be rare. the acquired for

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



FINAL RAG ANSWER
The retrieved evidence is insufficient to answer this question reliably. The provided chunks discuss topics related to diabetes, such as diabetic neuropathy, nephrogenic diabetes insipidus, neurohypophyseal diabetes insipidus, and neonatal diabetes mellitus, but none of the evidence directly addresses the symptoms of diabetes.


In [15]:
while True:
    question = input("\nEnter a medical question (or 'exit'): ").strip()

    if question.lower() in {"exit", "quit", "q"}:
        print("Interactive RAG session ended.")
        break

    if not question:
        print("Please enter a non-empty question.")
        continue

    try:
        _ = ask_medical_rag(
            question,
            top_k=TOP_K,
            show_context=True,
        )
    except Exception as e:
        print(f"Error: {type(e).__name__}: {e}")


Enter a medical question (or 'exit'): What is the primary reason of aids

USER QUESTION
What is the primary reason of aids

RETRIEVED EVIDENCE

[Chunk 1] cosine=0.7017 | row=9522 | chunk=0
Original MedQuAD question: What is the outlook for Neurological Complications of AIDS ?
Focus area: Neurological Complications of AIDS
Source: NINDS
Text:
the overall prognosis for individuals with aids in recent years has improved significantly because of new drugs and treatments. aids clinicians often fail to recognize neurological complications of aids. those who suspect they are having neurological complications should be sure to discuss these with their doctor.
--------------------------------------------------------------------------------------------------------------

[Chunk 2] cosine=0.6983 | row=9523 | chunk=0
Original MedQuAD question: what research (or clinical trials) is being done for Neurological Complications of AIDS ?
Focus area: Neurological Complications of AIDS
Source: NINDS
Text

KeyboardInterrupt: Interrupted by user

## Compact result table

In [16]:
def retrieved_results_table(results):
    rows = []

    for r in results:
        rows.append({
            "chunk": f"Chunk {r['rank']}",
            "cosine_score": round(r["score"], 4),
            "focus_area": r.get("focus_area"),
            "original_question": r.get("question"),
            "source": r.get("source"),
            "row_id": r.get("row_id"),
            "chunk_id": r.get("chunk_id"),
            "text": r.get("text"),
        })

    return pd.DataFrame(rows)